In [43]:
import pandas as pd
import numpy as np
import sys
parent_path = '/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline'
sys.path.append(parent_path)

from SynOmics.processing.metadata import MetaData
from SynOmics.processing.postprocessing import post_masking
from typing import Dict, Optional, Tuple
from SynOmics.metrics.fidelity.PairwiseSimilarity import PairwiseSimilarity
from SynOmics.utils.correlations import (
       _float_to_int_labels_with_nan,
       _compute_dense_labels_for_column,
       cramers_v_bincount_numba)


In [44]:
seed  = 42
original_data = pd.read_csv("../Data/original_data.csv", index_col = 0).iloc[:,0:500]
synthetic_data = pd.read_csv('../Data/avatarsk5_42.csv', index_col = 0).iloc[:,0:500]

masked_or_data = post_masking(original_data)
masked_syn_data = post_masking(synthetic_data)

ordinal_cat_columns = ["MSKCC", "Number_of_Prior_Therapies", "ORR","ExtremeResponder","Benefit"]
metadata = MetaData.get_metadata(data = masked_or_data, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ordinal_cat_columns)

ps = PairwiseSimilarity(
        original_data = masked_or_data,
        synthetic_data = masked_syn_data,
        metadata = metadata,
        output_dir = f"PW_Clinical_Omics/PairwiseClinical_{seed}",
        verbose = True,
        save = True,
        name = 'cross_group_analysis'
    )

In [46]:
or_processed_data = ps._process(masked_or_data, metadata)
syn_processed_data = ps._process(masked_syn_data, metadata)

In [53]:

# Dữ liệu ban đầu dạng float (như trong DataFrame)
feature_1_float = or_processed_data.loc[:,'Cohort'].values
feature_2_float = or_processed_data.loc[:,'ENSG00000227232.5'].values

# Bước 1: Convert float sang int labels (NaN -> -1)
feature_1_int = _float_to_int_labels_with_nan(feature_1_float)
feature_2_int = _float_to_int_labels_with_nan(feature_2_float)

# Bước 2: Convert sang dense labels
dense_f1, n_unique_f1 = _compute_dense_labels_for_column(feature_1_int)
dense_f2, n_unique_f2 = _compute_dense_labels_for_column(feature_2_int)

# Bước 3: Tính Cramér's V
cramers_v = cramers_v_bincount_numba(
    dense_f1.astype(np.int32),
    dense_f2.astype(np.int32),
    int(n_unique_f1),
    int(n_unique_f2)
)

print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.1341


In [48]:
dict_results = ps.get_pairwise_scores(method="spearman")

--- System & Process Info ---
Current Date and Time (UTC): 2026-02-08 16:35:31
Current User's Login: trinhtc
CPU Model: x86_64
Physical Cores: 40
Logical Processors: 40
Process RAM before execution: 607.75 MB

--- Function Execution ---
Processing data
Calculating correlation matrix for both original data and synthetic data. This matrix is mixed between spearman correlation and Cramér's V.

Summary of Pairwise score:
Summary of Pairwise Score Matrix:
  Number of elements: 121771
  Min value: 0.0949
  Max value: 1.0000
  Mean: 0.9376
  Median: 0.9726
  Standard deviation: 0.1116

--- Resource Usage Summary ---
Execution time: 10.197415 seconds
Process RAM after execution: 638.60 MB
Process RAM used by function: 30.85 MB
Average per-core CPU during execution: [0.0, 4.2, 0.0, 4.1, 0.4, 16.7, 0.0, 10.7, 0.0, 4.4, 0.0, 9.4, 2.1, 9.3, 0.9, 4.3, 0.4, 2.9, 0.0, 4.2, 0.1, 0.0, 0.2, 0.0, 0.1, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
CPU Cores utilized (>10%): 2 

In [49]:
clinical_features = masked_or_data.columns.tolist()[0:52]
transcriptomic_features = masked_or_data.columns.tolist()[52:]

In [50]:
cross_group_results = ps.get_cross_group_associations(
    group_1_features=clinical_features,
    group_2_features=transcriptomic_features,
    score_matrix=dict_results['PairwiseScore'],
    original_matrix=dict_results['OriginalCorrelation'],
    synthetic_matrix=dict_results['SyntheticCorrelation'],
    condensed=True
)

--- System & Process Info ---
Current Date and Time (UTC): 2026-02-08 16:36:24
Current User's Login: trinhtc
CPU Model: x86_64
Physical Cores: 40
Logical Processors: 40
Process RAM before execution: 638.60 MB

--- Function Execution ---
Computing 22984 cross-group associations:
  Group 1 (52 features): ['Cohort', 'Arm', 'Sex']...
  Group 2 (442 features): ['ENSG00000223972.5', 'ENSG00000227232.5', 'ENSG00000284332.1']...
--- System & Process Info ---
Current Date and Time (UTC): 2026-02-08 16:36:24
Current User's Login: trinhtc
CPU Model: x86_64
Physical Cores: 40
Logical Processors: 40
Process RAM before execution: 640.30 MB

--- Function Execution ---

--- Resource Usage Summary ---
Execution time: 0.052639 seconds
Process RAM after execution: 642.42 MB
Process RAM used by function: 2.12 MB
Average per-core CPU during execution: [0.0, 0.0, 0.0, 0.0, 100.0, 0.0, 0.0, 100.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

In [51]:
cross_group_results

,Feature_1,Feature_2,Metrics,Original_Correlation,Synthetic_Correlation,Score
0,Cohort,ENSG00000223972.5,CramersV_Correlation,0.668461,0.208814,0.540353
1,Cohort,ENSG00000227232.5,CramersV_Correlation,0.156608,0.172650,0.983958
2,Cohort,ENSG00000284332.1,CramersV_Correlation,0.657285,0.215404,0.558119
3,Cohort,ENSG00000237613.2,CramersV_Correlation,0.083222,0.155775,0.927447
4,Cohort,ENSG00000268020.3,CramersV_Correlation,1.000000,0.616792,0.616792
...,...,...,...,...,...,...
22979,ZNF800,ENSG00000200884.1,CramersV_Correlation,0.142071,0.139771,0.997700
22980,ZNF800,ENSG00000200351.1,CramersV_Correlation,0.142071,0.139771,0.997700
22981,ZNF800,ENSG00000199378.1,CramersV_Correlation,0.142071,0.139771,0.997700
22982,ZNF800,ENSG00000207404.1,CramersV_Correlation,0.142071,0.139771,0.997700
